## HANDOFF NOTE — read before touching this notebook further

**Status as of last edit:** chat-template fix for cell 7
(`make_multi_cycle_stream_fast`) is code-complete but **UNVERIFIED** --
it has never been executed (this environment has no GPU/torch). One
known follow-up change is still required and NOT yet applied. Do not
run the full sweep (last sweep cell) until the smoke test cell passes
cleanly.

### Chain of bugs found and fixed, in order

1. **Position-collision bug (FIXED, cell "RSQR arm" -- `run_strategy_c_rsqr`):**
   `logical_pos` was doing double duty as both the model's absolute
   position counter AND the eviction-compaction shift amount, so it went
   flat across repeated evictions (e.g. stuck at 45 for several
   consecutive FLAG events in early debug logs) -- meaning multiple
   distinct tokens were silently assigned the SAME RoPE position. This
   explained accuracy collapsing as `n_cycles`/eviction count grew.
   Fixed by splitting into `true_pos` (monotonic, never decremented,
   used for every `model_step` call) and `logical_pos` (derived as
   `true_pos_at_capture - cumulative_shift`, used only to compute
   survivors' rotation targets). **Verified via smoke-test log:**
   `true_pos` now climbs strictly through every eviction with no flat
   spots.

2. **Task-format bug (CODE WRITTEN, NOT YET VERIFIED --
   `make_multi_cycle_stream_fast`, cell after `tag_query_prefix`):**
   After fixing #1, accuracy was STILL near-random and specifically
   biased toward predicting digit token "1" (id 16) almost every trial
   regardless of the actual magic number or eviction strategy (A/B/C
   all equally affected). Root-caused via direct, zero-eviction,
   zero-cache-hack forward pass: the raw completion-style prompt
   ("The secret number is 5. The secret number is" + score next token)
   does NOT elicit digit completion from Qwen2.5-0.5B-Instruct at all --
   top1 prediction was a space/"the"/"a", never a digit, on a clean
   context with the fact immediately adjacent. This is a prompt-FORMAT
   problem (instruct models expect chat-template structure), not a
   capacity problem -- confirmed separately: Qwen2.5-1.5B-Instruct on
   the SAME raw prompt also failed (still no digit at top1), ruling out
   "just use a bigger model" as a fix. Wrapping the identical fact in
   real chat-template tokens (system preamble + user turn stating fact
   + explicit question "What is the secret number? Answer with only
   the digit." + assistant-turn-open) WAS confirmed working in
   isolation: correct digit becomes rank 1 with a real logit margin
   (23.63 vs 23.32 for the next-closest).

   `make_multi_cycle_stream_fast` was rewritten to splice real
   chat-template special tokens (`SYSTEM_PREFIX_IDS`,
   `ASSISTANT_OPEN_IDS`, `_QUESTION_IDS` -- all derived by rendering
   `tokenizer.apply_chat_template` ONCE with a placeholder marker and
   locating token spans, rather than hand-writing special-token ids)
   around the same fact/filler content as before. This code has NEVER
   been run -- no torch/GPU available in the environment that wrote it.

### REQUIRED next step before running anything

Every hardcoded `sink_size=6` call-site argument (smoke test cell,
sweep cell, and possibly cell 8/9's own defaults) must be changed to
`sink_size=len(SYSTEM_PREFIX_IDS)`. The old `sink_size=6` assumed 6
random filler tokens; the new sink region is the REAL system-preamble
token sequence, which is almost certainly longer than 6 tokens. If this
isn't fixed, sink-filling will stop after 6 tokens and the TAIL of the
real system preamble will be silently misrouted into the
fact-detection branch (`elif n_fact_tokens_seen < fact_len`), corrupting
`fact_caches` in a way that would be hard to notice from the outside
(no crash, just wrong attention content).

**This has NOT been done yet.** Find every `sink_size=6` in the
notebook and replace with `sink_size=len(SYSTEM_PREFIX_IDS)` before
running the smoke test.

### How to verify the fix actually worked

1. Run the cell defining `SYSTEM_PREFIX_IDS`/`ASSISTANT_OPEN_IDS`/
   `_QUESTION_IDS` first -- it prints decoded text for each; sanity
   check these look like real `<|im_start|>system...<|im_end|>` etc.,
   not garbled.
2. Fix all `sink_size=6` occurrences as described above.
3. Run the smoke-test cell. Check:
   - No assertion errors (stream format / pinning assertions).
   - `true_pos` still climbs strictly in the log output (regression
     check for bug #1).
   - Crucially: does `C RSQR`, `B corrected`, and `B uncorrected` now
     predict something OTHER than token 16 ("1") when it's wrong? A
     few wrong answers are fine/expected: what would indicate bug #2
     is NOT actually fixed is systematic convergence on one specific
     wrong digit across most/all trials, same as before.
4. Only after the smoke test looks sane (varied wrong answers, not one
   dominant wrong answer, `true_pos` monotonic) should the full 7-point
   sweep be run -- it takes multiple hours per the earlier broken run's
   timing.

### Known remaining open design question (not a bug, a decision)

`rotate_survivors_from_raw` implements KEY-side rotation-from-raw, not
QUERY-side as RFC §3.1 specifies. This is documented as deliberate in
that cell's docstring: `model_step`'s attention call applies ONE query
against the FULL concatenated key cache in a single matmul, so true
per-region query-side correction would require splitting attention
into multiple matmuls (one per differently-shifted key region) -- a
real architecture change not implemented here. Key-side
rotation-from-raw is mathematically equivalent for a single q-k pair
(RoPE scores depend only on relative position), so this is a legitimate
implementation choice, not an error -- but worth knowing if comparing
results back against the RFC's literal §3.1 wording.

### Precision findings (separately validated, NOT related to any bug above)

These came from a different notebook/session (isolated RoPE tensor
sweeps, no model, no eviction harness) and remain valid regardless of
the bugs above:
- Drift does not compound across repeated eviction cycles -- bounded
  oscillation (8e-6 to 3e-5 range across 100 iterations), not growth.
- The magnitude-driven precision floor is driven by `evict_n` size, not
  by `P` or by target position (`P - evict_n`) individually.
- This floor is invisible past softmax (prob diff ~1e-6, same order as
  fp32 noise) -- doesn't propagate into attention-score differences at
  a level that would explain the accuracy problems found in THIS
  notebook. In other words: the accuracy failures diagnosed above are
  NOT a rotation-precision issue -- they were a position-bookkeeping bug
  (#1) and a prompt-format bug (#2), both unrelated to RoPE's numerical
  precision, which was already shown to be solid.


---

## UPDATE — suffix-format bug found and fixed (supersedes bug #2 above)

Bug #2's fix (chat-template question suffix) was applied and run --
and it did NOT work. Direct forward-pass check (see the now-historical
probe cell) showed top1 was a space token, correct digit absent from
top5 entirely, even with `sink_size` correctly wired to
`len(SYSTEM_PREFIX_IDS)`. The chat-template-necessity theory was
half right and half wrong: wrapping the fact/filler content in a real
system/user chat turn is fine and harmless, but ending the prompt on
an explicit question + "answer with only the digit" instruction does
NOT reliably elicit digit completion from this model once filler
content is in between -- confirmed directly, not theorized.

**Root cause, isolated directly:** the ORIGINAL notebook's plain
unfinished-sentence completion style ("The secret number is" + score
next token, no question, no instruction) still works fine wrapped in
the same chat-template prefix -- 3/3 correct@rank1 across
n_cycles=2/4/12. So the actual fix was removing the question/instruction
framing entirely, not adding more chat-template structure.

**Applied fix:** `make_multi_cycle_stream_fast` (cell defining
`SYSTEM_PREFIX_IDS`) now uses `SYSTEM_PREFIX_IDS` as the pinned prefix
(unchanged, this part was never the problem) and a plain completion
fragment (`". The secret number is"` + trailing space, scored
directly) as the pinned suffix, in place of
`ASSISTANT_OPEN_IDS + _QUESTION_IDS`. `ASSISTANT_OPEN_IDS` and
`_QUESTION_IDS` no longer exist as names -- if any cell below still
references them, that cell is stale and needs the same suffix swap.

**Status: this fix has been applied but not yet re-verified end to end
in this exact file** (the smoke-test cell and full sweep cell need a
fresh run against the corrected `make_multi_cycle_stream_fast`).
Before trusting sweep numbers:
1. Re-run the cell defining `SYSTEM_PREFIX_IDS`/`_QUERY_SUFFIX_IDS` --
   check the sanity decodes look right.
2. Re-run the smoke-test cell. Same checks as before (`true_pos`
   monotonic, stream format assertions pass) PLUS: check the printed
   `top5_ids`/predicted token actually vary sensibly across trials
   (not all collapsing to one wrong token, not all collapsing to a
   space).
3. Only then run the full sweep.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
torch.manual_seed(0)
device = "cuda:0"


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # real Llama-arch model, small enough for CPU


In [ ]:
# Set the device dynamically
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Targeting Device: {device.upper()}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
hf_model.eval()

cfg = hf_model.config
N_LAYERS = cfg.num_hidden_layers
N_HEADS = cfg.num_attention_heads
N_KV_HEADS = getattr(cfg, "num_key_value_heads", N_HEADS)
HEAD_DIM = cfg.hidden_size // N_HEADS
KV_GROUPS = N_HEADS // N_KV_HEADS

# ROPE_THETA: check the plain attribute first, fall back to the nested
# rope_scaling dict (where Qwen and some other configs actually put it).
# No intermediate "wrong default" step -- this is correct on first build.
ROPE_THETA = getattr(cfg, "rope_theta", None)
if ROPE_THETA is None and getattr(cfg, "rope_scaling", None):
    ROPE_THETA = cfg.rope_scaling.get("rope_theta")
assert ROPE_THETA is not None, "couldn't find rope_theta anywhere on cfg -- inspect cfg directly"

def rope_freqs(dim, base):
    assert dim % 2 == 0
    i = torch.arange(0, dim // 2, dtype=torch.float32)
    return base ** (-2.0 * i / dim)

FREQS = rope_freqs(HEAD_DIM, ROPE_THETA)

print(f"layers={N_LAYERS} heads={N_HEADS} kv_heads={N_KV_HEADS} head_dim={HEAD_DIM} "
      f"rope_theta={ROPE_THETA} model_device={next(hf_model.parameters()).device}")

# Self-check every time this cell runs -- if FREQS[1] ever drifts from what
# ROPE_THETA implies (e.g. from a stray edit reintroducing a hardcoded base),
# this fails loudly instead of silently producing wrong numbers again.
expected = ROPE_THETA ** (-2.0 / HEAD_DIM)
actual = FREQS[1].item()
assert abs(expected - actual) < 1e-4, f"FREQS/ROPE_THETA mismatch: expected {expected}, got {actual}"
print(f"FREQS verified against ROPE_THETA={ROPE_THETA}: OK")


Targeting Device: CUDA


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

layers=24 heads=14 kv_heads=2 head_dim=64 rope_theta=1000000.0 model_device=cuda:0
FREQS verified against ROPE_THETA=1000000.0: OK


In [ ]:
def rope_freqs(dim, base):
    assert dim % 2 == 0
    i = torch.arange(0, dim // 2, dtype=torch.float32)
    return base ** (-2.0 * i / dim)
FREQS = rope_freqs(HEAD_DIM, ROPE_THETA)
def apply_rope(x, positions, freqs):
    """x: [..., T, Dh]; positions: [T]."""
    # Dynamically find the active hardware backend (e.g., 'cuda:0')
    current_device = x.device

    # Cast positions and frequencies to match the target device
    if not isinstance(positions, torch.Tensor):
        positions = torch.tensor(positions, device=current_device)
    else:
        positions = positions.to(current_device)

    freqs = freqs.to(current_device) if isinstance(freqs, torch.Tensor) else freqs

    half = x.shape[-1] // 2
    x1, x2 = x[..., :half], x[..., half:]

    # Position embedding math now runs entirely on the correct device
    angles = positions[:, None] * freqs[None, :]
    cos = torch.cos(angles).to(x.dtype)
    sin = torch.sin(angles).to(x.dtype)
    while cos.dim() < x.dim():
        cos = cos.unsqueeze(0)
        sin = sin.unsqueeze(0)

    out1 = x1 * cos - x2 * sin
    out2 = x2 * cos + x1 * sin
    return torch.cat([out1, out2], dim=-1)

def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    b, h, t, d = x.shape
    x = x[:, :, None, :, :].expand(b, h, n_rep, t, d)
    return x.reshape(b, h * n_rep, t, d)


In [ ]:
@torch.no_grad()
def layer_step(layer, x, position, kv_cache):
    """x: [1,1,D] single new token's hidden state. kv_cache: dict with
    'k','v' shape [1, n_kv_heads, Tc, Dh] or None. Returns (new_x, new_kv_cache)."""
    attn = layer.self_attn
    residual = x
    h = layer.input_layernorm(x)
    q = attn.q_proj(h).view(1, 1, N_HEADS, HEAD_DIM).transpose(1, 2)
    k = attn.k_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    v = attn.v_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    pos_t = torch.tensor([position], dtype=torch.float32)
    q = apply_rope(q, pos_t, FREQS)
    k = apply_rope(k, pos_t, FREQS)
    if kv_cache is not None and kv_cache["k"].shape[2] > 0:
        k_full = torch.cat([kv_cache["k"], k], dim=2)
        v_full = torch.cat([kv_cache["v"], v], dim=2)
    else:
        k_full, v_full = k, v
    k_rep = repeat_kv(k_full, KV_GROUPS)
    v_rep = repeat_kv(v_full, KV_GROUPS)
    att = (q @ k_rep.transpose(-2, -1)) / (HEAD_DIM ** 0.5)
    att = F.softmax(att, dim=-1)
    out = att @ v_rep
    out = out.transpose(1, 2).reshape(1, 1, N_HEADS * HEAD_DIM)
    out = attn.o_proj(out)
    x = residual + out
    residual = x
    h = layer.post_attention_layernorm(x)
    x = residual + layer.mlp(h)
    return x, {"k": k_full, "v": v_full}
@torch.no_grad()
def model_step(token_id, position, kv_caches):
    """token_id: python int. kv_caches: list of per-layer dicts or Nones.
    Returns (logits[vocab], new_kv_caches)."""
        # Dynamically find the active device of the model parameters (e.g., 'cuda:0')
    current_device = next(hf_model.parameters()).device

    # Force the input token tensor to be created directly on the target hardware backend
    token_tensor = torch.tensor([[token_id]], device=current_device)

    # Process the embeddings on the proper device
    x = hf_model.model.embed_tokens(token_tensor)

    new_caches = []
    for layer, cache in zip(hf_model.model.layers, kv_caches):
        x, new_cache = layer_step(layer, x, position, cache)
        new_caches.append(new_cache)

    x = hf_model.model.norm(x)
    logits = hf_model.lm_head(x)[0, 0]
    return logits, new_caches

def empty_caches(n_layers):
    return [None] * n_layers
def concat_cache(a, b):
    if a is None or a["k"].shape[2] == 0:
        return b
    if b is None or b["k"].shape[2] == 0:
        return a
    return {"k": torch.cat([a["k"], b["k"]], dim=2), "v": torch.cat([a["v"], b["v"]], dim=2)}
def slice_cache(cache, start=0, end=None):
    if cache is None:
        return None
    return {"k": cache["k"][:, :, start:end, :], "v": cache["v"][:, :, start:end, :]}
@torch.no_grad()
def rerotate_cache(kv_caches, delta):
    """Exact corrective rotation applied to cached K only -- never V, never sink."""
    out = []
    for cache in kv_caches:
        if cache is None or cache["k"].shape[2] == 0:
            out.append(cache)
            continue
        Tc = cache["k"].shape[2]
        delta_t = torch.full((Tc,), float(delta))
        out.append({"k": apply_rope(cache["k"], delta_t, FREQS), "v": cache["v"]})
    return out


In [ ]:
FILLER_WORDS = ["the", "cat", "sat", "on", "mat", "and", "ran", "far", "away",
                "into", "town", "market", "river", "bridge", "forest", "path"]


In [ ]:
def tag_query_prefix(trial_stream, query_prefix_len):
    """Wraps a (tid, is_query, answer_id) stream into
    (tid, is_query, answer_id, is_pinned) tuples, where is_pinned=True for
    the final `query_prefix_len` tokens PLUS the is_query token itself --
    exactly the run that must never be routed into window_tokens /
    eviction / flagging bookkeeping in any strategy.

    BUG THIS FIXES: every strategy (A, B, C) previously treated
    query-prefix tokens (e.g. "Alice's", "secret", "number", "is") as
    ordinary window/filler tokens for every step except the final
    is_query=True one. That meant they could be silently evicted -- or,
    in RSQR's case, flagged into the shadow store -- before the model
    ever saw the actual query token, corrupting exactly the content the
    query depends on. This got worse as n_cycles grew, since a longer
    stream means more eviction/flagging events could land inside the
    query prefix's brief window lifetime.

    query_prefix_len is the token count of the query prefix text only
    (e.g. len(_QUERY_BASE_IDS) or len(query_base_ids) below) -- NOT
    including the trailing is_query token, which is always pinned
    regardless.
    """
    n = len(trial_stream)
    pin_from = n - 1 - query_prefix_len  # index of first pinned prefix token
    return [
        (tid, is_query, answer_id, idx >= pin_from)
        for idx, (tid, is_query, answer_id) in enumerate(trial_stream)
    ]


In [ ]:
FILLER_WORD_IDS = {w: tokenizer(" " + w, add_special_tokens=False)["input_ids"][0] for w in FILLER_WORDS}

# SUFFIX FORMAT FIX (supersedes the chat-template question suffix below):
# cell 18's direct-forward-pass check showed that even with sink_size
# correctly set to len(SYSTEM_PREFIX_IDS) and a real chat-template
# system/user preamble in place, ending the prompt on
# ASSISTANT_OPEN_IDS + "What is the secret number? Answer with only the
# digit:" still does NOT elicit digit completion -- top1 was a space
# (id 220), with the correct digit (id 22) absent from top5 entirely.
# The question/instruction framing was itself the problem, independent
# of chat-template wrapping.
#
# cell 19 isolated this directly: keeping the SAME chat-wrapped
# system/user prefix and the SAME filler/fact content, but replacing
# the suffix with a plain unfinished-sentence completion ("The secret
# number is" + score next token, no question, no instruction -- the
# ORIGINAL notebook's exact suffix style) got the correct digit to
# rank 1 in 3/3 trials across n_cycles=2/4/12, matching the original
# notebook's reliability. So the fix is not "add more chat-template
# structure" -- it's "don't ask a question at all"; a system/user
# preamble is harmless to keep (matches how the model was tuned) but
# the query itself needs to stay in plain-completion form.
#
# We keep SYSTEM_PREFIX_IDS as the sink/pinned prefix (real
# <|im_start|>system...<|im_end|><|im_start|>user\n tokens -- this
# part was never the problem) and replace ONLY the suffix: instead of
# ASSISTANT_OPEN_IDS + _QUESTION_IDS, we use the plain completion
# fragment ". The secret number is" and score the next token directly,
# exactly as cell 19 confirmed working.

_SYSTEM_PREAMBLE_TEXT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

_probe_messages = [
    {"role": "system", "content": _SYSTEM_PREAMBLE_TEXT},
    {"role": "user", "content": "@@USERCONTENT@@"},
]
_probe_prompt = tokenizer.apply_chat_template(_probe_messages, tokenize=False, add_generation_prompt=True)
_marker_ids = tokenizer("@@USERCONTENT@@", add_special_tokens=False)["input_ids"]
_full_ids = tokenizer(_probe_prompt, add_special_tokens=False)["input_ids"]

def _find_subsequence(haystack, needle):
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i:i+m] == needle:
            return i
    raise ValueError("marker token sequence not found in rendered chat template -- "
                      "tokenizer may be splitting @@USERCONTENT@@ differently than expected")

_marker_start = _find_subsequence(_full_ids, _marker_ids)
_marker_end = _marker_start + len(_marker_ids)

SYSTEM_PREFIX_IDS = _full_ids[:_marker_start]        # <|im_start|>system ... <|im_end|>\n<|im_start|>user\n
_SPACE_ID = tokenizer(" ", add_special_tokens=False)["input_ids"][0]
_QUERY_SUFFIX_IDS = tokenizer(". The secret number is", add_special_tokens=False)["input_ids"]

print(f"SYSTEM_PREFIX_IDS: {len(SYSTEM_PREFIX_IDS)} tokens")
print(f"QUERY_SUFFIX_IDS: {len(_QUERY_SUFFIX_IDS)} tokens")
print("sanity decode of SYSTEM_PREFIX_IDS:", repr(tokenizer.decode(SYSTEM_PREFIX_IDS)))
print("sanity decode of QUERY_SUFFIX_IDS:", repr(tokenizer.decode(_QUERY_SUFFIX_IDS)))


def make_multi_cycle_stream_fast(rng, n_cycles, cycle_len, sink_size=None):
    """Chat-wrapped prefix + plain-completion suffix (see fix note above).

    Stream layout (all as plain (tid, is_query, answer_id) tuples
    before tag_query_prefix wraps them with is_pinned):

      [SYSTEM_PREFIX_IDS]                    -- pinned prefix. Real
          <|im_start|>system ... <|im_end|><|im_start|>user\n tokens.
          sink_size is unused for content selection (kept as a
          parameter for call-site compatibility); the actual
          pinned-prefix length is len(SYSTEM_PREFIX_IDS).
      [magic fact tokens]                    -- eviction-eligible
      [n_cycles * cycle_len filler tokens]   -- eviction-eligible
      [_QUERY_SUFFIX_IDS]                    -- pinned suffix, plain
          completion fragment (". The secret number is"), then a
          trailing space token is appended and scored directly --
          exactly the ORIGINAL notebook's query style, confirmed
          working in cell 19.
    """
    if sink_size is None:
        sink_size = len(SYSTEM_PREFIX_IDS)
    magic_number = rng.integers(2, 9)
    magic_ids = tokenizer(f"The secret number is {magic_number}.", add_special_tokens=False)["input_ids"]
    answer_id = tokenizer(str(magic_number), add_special_tokens=False)["input_ids"][0]

    words = list(FILLER_WORD_IDS.keys())
    ids = [FILLER_WORD_IDS[w] for w in words]

    stream = [(tid, False, None) for tid in SYSTEM_PREFIX_IDS]
    stream += [(tid, False, None) for tid in magic_ids]

    n_filler = n_cycles * cycle_len
    for i in rng.integers(0, len(words), size=n_filler):
        stream.append((ids[i], False, None))

    for tid in _QUERY_SUFFIX_IDS:
        stream.append((tid, False, None))
    stream.append((_SPACE_ID, True, answer_id))

    # query_prefix_len covers _QUERY_SUFFIX_IDS only -- the trailing
    # space/is_query token is always pinned regardless (tag_query_prefix's
    # own contract), so it's not counted here.
    query_prefix_len = len(_QUERY_SUFFIX_IDS)
    stream = tag_query_prefix(stream, query_prefix_len=query_prefix_len)

    fact_len = len(magic_ids)
    return stream, fact_len, magic_number


SYSTEM_PREFIX_IDS: 24 tokens
QUERY_SUFFIX_IDS: 5 tokens
sanity decode of SYSTEM_PREFIX_IDS: '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n'
sanity decode of QUERY_SUFFIX_IDS: '. The secret number is'


In [ ]:
@torch.no_grad()
def run_strategy_a_fact_persistence(trial_stream, fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, horizon=48):
    sink_caches = empty_caches(N_LAYERS)
    fact_caches = empty_caches(N_LAYERS)
    window_caches = empty_caches(N_LAYERS)
    pinned_caches = empty_caches(N_LAYERS)
    sink_tokens, fact_tokens, window_tokens, pinned_tokens = [], [], [], []
    true_pos = 0
    n_fact_tokens_seen = 0
    result = {"correct": None, "answer_logprob": None, "n_consolidations": 0, "tokens_reprocessed": 0}

    def replay_all(sink_ids, fact_ids, window_ids):
        caches = empty_caches(N_LAYERS)
        pos = 0
        for tid in sink_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        n_sink = len(sink_ids)
        sink_c = [slice_cache(c, start=0, end=n_sink) for c in caches]
        for tid in fact_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        n_fact = len(fact_ids)
        fact_c = [slice_cache(c, start=n_sink, end=n_sink + n_fact) for c in caches]
        for tid in window_ids:
            pos += 1
            _, caches = model_step(tid, pos, caches)
        window_c = [slice_cache(c, start=n_sink + n_fact) for c in caches]
        result["tokens_reprocessed"] += len(sink_ids) + len(fact_ids) + len(window_ids)
        return sink_c, fact_c, window_c, pos

    # NOTE: replay_all above only reconstructs sink/fact/window (no pinned
    # region) -- this is fine because consolidation only ever fires on a
    # window-token step (the horizon check lives in the else branch below),
    # and the stream format guarantees no window token ever follows a
    # pinned one, so pinned_tokens is always empty whenever replay_all runs.

    for tid, is_query, answer_id, is_pinned in trial_stream:
        if is_query:
            base = [concat_cache(s, concat_cache(f, concat_cache(p, w)))
                    for s, f, p, w in zip(sink_caches, fact_caches, pinned_caches, window_caches)]
            true_pos += 1
            logits, _ = model_step(tid, true_pos, base)
            true_pos -= 1
            result["correct"] = (logits.argmax(-1).item() == answer_id)
            result["answer_logprob"] = F.log_softmax(logits, dim=-1)[answer_id].item()
            continue

        true_pos += 1
        base = [concat_cache(s, concat_cache(f, concat_cache(p, w)))
                for s, f, p, w in zip(sink_caches, fact_caches, pinned_caches, window_caches)]

        if len(sink_tokens) < sink_size:
            sink_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            sink_caches = [slice_cache(c, start=0, end=len(sink_tokens)) for c in new_full]

        elif n_fact_tokens_seen < fact_len:
            fact_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact_prior = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            fact_caches = [slice_cache(c, start=n_sink, end=n_sink + n_fact_prior + 1) for c in new_full]
            n_fact_tokens_seen += 1

        elif is_pinned:
            # Query-prefix token: appended to the pinned region, which
            # sits BETWEEN fact and window in concat order. Window's own
            # offset math below always re-reads n_pinned fresh each step,
            # so it stays correct regardless of how much pinned content
            # has accumulated.
            pinned_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            pinned_caches = [slice_cache(c, start=n_sink + n_fact, end=n_sink + n_fact + len(pinned_tokens)) for c in new_full]

        else:
            window_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            n_pinned = pinned_caches[0]["k"].shape[2] if pinned_caches[0] is not None else 0
            window_caches = [slice_cache(c, start=n_sink + n_fact + n_pinned, end=n_sink + n_fact + n_pinned + len(window_tokens)) for c in new_full]

            if len(window_tokens) > window_size:
                window_tokens.pop(0)
                window_caches = [slice_cache(c, start=1) for c in window_caches]

            dist = true_pos - (sink_size + fact_len - 1)
            if dist >= horizon:
                sink_caches, fact_caches, window_caches, true_pos = replay_all(
                    sink_tokens, fact_tokens, window_tokens
                )
                result["n_consolidations"] += 1

    return result


In [ ]:
@torch.inference_mode()
def _run_prefix_until_first_eviction(trial_stream, fact_len, sink_size, window_size, block_size):
    sink_caches = empty_caches(N_LAYERS)
    fact_caches = empty_caches(N_LAYERS)
    window_caches = empty_caches(N_LAYERS)
    pinned_caches = empty_caches(N_LAYERS)
    sink_tokens, window_tokens, pinned_tokens = [], [], []
    logical_pos = 0
    n_fact_tokens_seen = 0

    for idx, (tid, is_query, answer_id, is_pinned) in enumerate(trial_stream):
        if is_query:
            return dict(done=True, sink_caches=sink_caches, fact_caches=fact_caches,
                        window_caches=window_caches, window_tokens=window_tokens,
                        pinned_caches=pinned_caches, pinned_tokens=pinned_tokens,
                        logical_pos=logical_pos, resume_idx=idx, n_evictions=0)

        logical_pos += 1
        base = [concat_cache(s, concat_cache(f, concat_cache(p, w)))
                for s, f, p, w in zip(sink_caches, fact_caches, pinned_caches, window_caches)]

        if len(sink_tokens) < sink_size:
            sink_tokens.append(tid)
            _, new_full = model_step(tid, logical_pos, base)
            sink_caches = [slice_cache(c, start=0, end=len(sink_tokens)) for c in new_full]
        elif n_fact_tokens_seen < fact_len:
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact_prior = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            fact_caches = [slice_cache(c, start=n_sink, end=n_sink + n_fact_prior + 1) for c in new_full]
            n_fact_tokens_seen += 1
        elif is_pinned:
            pinned_tokens.append(tid)
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            pinned_caches = [slice_cache(c, start=n_sink + n_fact, end=n_sink + n_fact + len(pinned_tokens)) for c in new_full]
        else:
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            n_pinned = pinned_caches[0]["k"].shape[2] if pinned_caches[0] is not None else 0
            window_tokens.append(tid)
            window_caches = [slice_cache(c, start=n_sink + n_fact + n_pinned) for c in new_full]

            if len(window_tokens) > 0 and len(window_tokens) % block_size == 0 and len(window_tokens) > window_size:
                evict_n = block_size
                window_tokens = window_tokens[evict_n:]
                window_caches = [slice_cache(c, start=evict_n) for c in window_caches]
                logical_pos -= evict_n
                # BUG FIX: this eviction was applied HERE, in the prefix
                # phase, but was previously never counted anywhere --
                # _run_from_state starts its own n_evictions at 0 and only
                # counts evictions it personally witnesses after the
                # handoff, so this one silently vanished from the total.
                # That desynced B's reported n_evictions from RSQR's by
                # exactly 1 for every trial where an eviction lands during
                # the shared prefix (which is the common case, since the
                # prefix phase runs until the FIRST eviction by
                # definition). The underlying cache/position state was
                # always correct -- only the reported count was wrong.
                # Fix: report n_evictions=1 here so the caller can fold it
                # into both arms' final totals.
                return dict(done=False, sink_caches=sink_caches, fact_caches=fact_caches,
                            window_caches=window_caches, window_tokens=window_tokens,
                            pinned_caches=pinned_caches, pinned_tokens=pinned_tokens,
                            logical_pos=logical_pos, resume_idx=idx + 1, n_evictions=1)

    return dict(done=True, sink_caches=sink_caches, fact_caches=fact_caches,
                window_caches=window_caches, window_tokens=window_tokens,
                pinned_caches=pinned_caches, pinned_tokens=pinned_tokens,
                logical_pos=logical_pos, resume_idx=idx, n_evictions=0)

@torch.inference_mode()
def _run_from_state(trial_stream, resume_idx, state, sink_size, window_size, block_size, apply_correction):
    sink_caches, fact_caches = state["sink_caches"], state["fact_caches"]
    window_caches = state["window_caches"]
    pinned_caches = state["pinned_caches"]
    window_tokens = list(state["window_tokens"])
    pinned_tokens = list(state["pinned_tokens"])
    logical_pos = state["logical_pos"]
    result = {"correct": None, "answer_logprob": None, "n_evictions": 0, "tokens_reprocessed": 0}

    # NOTE: the eviction that triggered the handoff into this function was
    # already applied by _run_prefix_until_first_eviction before it
    # returned (see fix above) -- so window_tokens/window_caches arriving
    # here are already post-eviction. This function's own eviction check
    # below only needs to handle evictions that occur AFTER the handoff,
    # same as before.
    for tid, is_query, answer_id, is_pinned in trial_stream[resume_idx:]:
        base = [concat_cache(s, concat_cache(f, concat_cache(p, w)))
                for s, f, p, w in zip(sink_caches, fact_caches, pinned_caches, window_caches)]
        if is_query:
            logical_pos += 1
            logits, _ = model_step(tid, logical_pos, base)
            logical_pos -= 1
            result["correct"] = (logits.argmax(-1).item() == answer_id)
            result["answer_logprob"] = F.log_softmax(logits, dim=-1)[answer_id].item()
            continue

        logical_pos += 1

        if is_pinned:
            pinned_tokens.append(tid)
            _, new_full = model_step(tid, logical_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            pinned_caches = [slice_cache(c, start=n_sink + n_fact, end=n_sink + n_fact + len(pinned_tokens)) for c in new_full]
            continue

        _, new_full = model_step(tid, logical_pos, base)
        n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
        n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
        n_pinned = pinned_caches[0]["k"].shape[2] if pinned_caches[0] is not None else 0
        window_tokens.append(tid)
        window_caches = [slice_cache(c, start=n_sink + n_fact + n_pinned) for c in new_full]

        if len(window_tokens) > 0 and len(window_tokens) % block_size == 0 and len(window_tokens) > window_size:
            evict_n = block_size
            n_survivors = window_caches[0]["k"].shape[2] - evict_n if window_caches[0] is not None else 0
            window_tokens = window_tokens[evict_n:]
            window_caches = [slice_cache(c, start=evict_n) for c in window_caches]
            if apply_correction:
                window_caches = rerotate_cache(window_caches, -evict_n)
                result["tokens_reprocessed"] += max(n_survivors, 0)
                logical_pos -= evict_n
            result["n_evictions"] += 1

    return result


def run_strategy_b_pair_fast(trial_stream, fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, block_size=8):
    """Returns (result_corrected, result_uncorrected), sharing all compute
    up to the first eviction instead of redoing it twice."""
    state = _run_prefix_until_first_eviction(trial_stream, fact_len, sink_size, window_size, block_size)

    if state["done"]:
        tid, is_query, answer_id, is_pinned = trial_stream[state["resume_idx"]]
        base = [concat_cache(s, concat_cache(f, concat_cache(p, w)))
                for s, f, p, w in zip(state["sink_caches"], state["fact_caches"],
                                       state["pinned_caches"], state["window_caches"])]
        logits, _ = model_step(tid, state["logical_pos"] + 1, base)
        r = {"correct": (logits.argmax(-1).item() == answer_id),
             "answer_logprob": F.log_softmax(logits, dim=-1)[answer_id].item(), "n_evictions": 0}
        return dict(r), dict(r)

    def clone_caches(caches):
        return [None if c is None else {"k": c["k"].clone(), "v": c["v"].clone()} for c in caches]

    state_c = dict(state, sink_caches=clone_caches(state["sink_caches"]),
                   fact_caches=clone_caches(state["fact_caches"]),
                   window_caches=clone_caches(state["window_caches"]),
                   pinned_caches=clone_caches(state["pinned_caches"]),
                   window_tokens=list(state["window_tokens"]),
                   pinned_tokens=list(state["pinned_tokens"]))
    state_u = dict(state, sink_caches=clone_caches(state["sink_caches"]),
                   fact_caches=clone_caches(state["fact_caches"]),
                   window_caches=clone_caches(state["window_caches"]),
                   pinned_caches=clone_caches(state["pinned_caches"]),
                   window_tokens=list(state["window_tokens"]),
                   pinned_tokens=list(state["pinned_tokens"]))

    # IMPORTANT: the eviction detected during the prefix phase has ALREADY
    # been applied uncorrected (no rerotate_cache) inside
    # _run_prefix_until_first_eviction, since that shared prefix, by
    # construction, is identical for both the corrected and uncorrected
    # arms up to this point -- there is nothing to correct yet because no
    # correction-relevant rotation has been skipped. Both branches
    # diverge (corrected applies rerotate_cache, uncorrected doesn't)
    # only for evictions from here onward, which is exactly what
    # apply_correction controls in _run_from_state below.
    r_c = _run_from_state(trial_stream, state["resume_idx"], state_c, sink_size, window_size, block_size, True)
    r_u = _run_from_state(trial_stream, state["resume_idx"], state_u, sink_size, window_size, block_size, False)
    # BUG FIX: the eviction applied during the shared prefix phase
    # (state["n_evictions"]) was never being counted in either arm's
    # final total -- see note in _run_prefix_until_first_eviction.
    r_c["n_evictions"] += state["n_evictions"]
    r_u["n_evictions"] += state["n_evictions"]
    return r_c, r_u


## RSQR arm: raw-survivor storage, query-side rotation


In [ ]:
import logging
import sys

# ---- RSQR verbose logging setup ----
logger = logging.getLogger("rsqr")
logger.setLevel(logging.INFO)   # <-- change to logging.DEBUG for full trace
logger.handlers.clear()
_handler = logging.StreamHandler(sys.stdout)
_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(_handler)
logger.propagate = False

def set_rsqr_log_level(level):
    logger.setLevel(level)
    print(f"rsqr logger level set to {logging.getLevelName(level)}")


In [ ]:
@torch.no_grad()
def layer_step_with_raw_capture(layer, x, position, kv_cache):
    """Identical to layer_step, but also returns the RAW (pre-RoPE) key
    AND the value (V is never rotated by RoPE)."""
    attn = layer.self_attn
    residual = x
    h = layer.input_layernorm(x)
    q = attn.q_proj(h).view(1, 1, N_HEADS, HEAD_DIM).transpose(1, 2)
    k_raw = attn.k_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    v = attn.v_proj(h).view(1, 1, N_KV_HEADS, HEAD_DIM).transpose(1, 2)
    pos_t = torch.tensor([position], dtype=torch.float32)
    q_rot = apply_rope(q, pos_t, FREQS)
    k_rot = apply_rope(k_raw, pos_t, FREQS)
    if kv_cache is not None and kv_cache["k"].shape[2] > 0:
        k_full = torch.cat([kv_cache["k"], k_rot], dim=2)
        v_full = torch.cat([kv_cache["v"], v], dim=2)
    else:
        k_full, v_full = k_rot, v
    k_rep = repeat_kv(k_full, KV_GROUPS)
    v_rep = repeat_kv(v_full, KV_GROUPS)
    att = (q_rot @ k_rep.transpose(-2, -1)) / (HEAD_DIM ** 0.5)
    att = F.softmax(att, dim=-1)
    out = att @ v_rep
    out = out.transpose(1, 2).reshape(1, 1, N_HEADS * HEAD_DIM)
    out = attn.o_proj(out)
    x = residual + out
    residual = x
    h = layer.post_attention_layernorm(x)
    x = residual + layer.mlp(h)
    return x, {"k": k_full, "v": v_full}, k_raw, v


@torch.no_grad()
def model_step_with_raw_capture(token_id, position, kv_caches):
    """Same as model_step, but also returns per-layer (raw_k, raw_v) for
    the new token."""
    current_device = next(hf_model.parameters()).device
    token_tensor = torch.tensor([[token_id]], device=current_device)
    x = hf_model.model.embed_tokens(token_tensor)

    new_caches, raw_ks, raw_vs = [], [], []
    for layer, cache in zip(hf_model.model.layers, kv_caches):
        x, new_cache, k_raw, v_raw = layer_step_with_raw_capture(layer, x, position, cache)
        new_caches.append(new_cache)
        raw_ks.append(k_raw)
        raw_vs.append(v_raw)

    x = hf_model.model.norm(x)
    logits = hf_model.lm_head(x)[0, 0]
    return logits, new_caches, raw_ks, raw_vs


class RawShadowStore:
    """Per-layer raw (unrotated) K + real V storage for flagged survivor
    tokens, keyed by the token's GLOBAL stream position (stable) rather
    than logical/compacted position (which shifts as eviction compacts
    the timeline, RFC §3.2 step 4)."""
    def __init__(self, n_layers):
        self.n_layers = n_layers
        self._store = {}
        self._n_adds = 0
        self._n_drops = 0

    def add(self, global_pos, raw_ks_per_layer, raw_vs_per_layer):
        self._store[global_pos] = {
            "k": [k.clone() for k in raw_ks_per_layer],
            "v": [v.clone() for v in raw_vs_per_layer],
        }
        self._n_adds += 1
        logger.debug(
            f"RawShadowStore.add global_pos={global_pos} "
            f"k_shape={tuple(raw_ks_per_layer[0].shape)} "
            f"v_shape={tuple(raw_vs_per_layer[0].shape)} "
            f"store_size={len(self._store)} (lifetime adds={self._n_adds})"
        )

    def drop(self, global_pos):
        existed = self._store.pop(global_pos, None) is not None
        if existed:
            self._n_drops += 1
            logger.debug(
                f"RawShadowStore.drop global_pos={global_pos} "
                f"store_size={len(self._store)} (lifetime drops={self._n_drops})"
            )
        else:
            logger.warning(
                f"RawShadowStore.drop called on global_pos={global_pos} "
                f"which was NOT in the store -- no-op, but check caller logic"
            )
        return existed

    def __len__(self):
        return len(self._store)

    def survivor_positions_sorted(self):
        return sorted(self._store.keys())

    def raw_kv_for_layer(self, layer_idx):
        positions = self.survivor_positions_sorted()
        if not positions:
            empty_k = torch.zeros(1, N_KV_HEADS, 0, HEAD_DIM)
            empty_v = torch.zeros(1, N_KV_HEADS, 0, HEAD_DIM)
            return [], empty_k, empty_v
        ks = torch.cat([self._store[p]["k"][layer_idx] for p in positions], dim=2)
        vs = torch.cat([self._store[p]["v"][layer_idx] for p in positions], dim=2)
        return positions, ks, vs

    def summary(self):
        return {"n_alive": len(self._store), "lifetime_adds": self._n_adds,
                "lifetime_drops": self._n_drops}


In [ ]:
@torch.no_grad()
def rotate_survivors_from_raw(raw_store, logical_positions_by_global_pos, layer_idx_for_log=0):
    """RFC §3.2 step 5 -- the core RSQR operation. Rotates each raw
    survivor key a SINGLE hop directly from raw to its CURRENT logical
    position (no correction-on-correction, no compounding drift path)."""
    out = []
    positions_for_log = None
    for layer_idx in range(raw_store.n_layers):
        global_positions, raw_k, raw_v = raw_store.raw_kv_for_layer(layer_idx)
        if layer_idx == layer_idx_for_log:
            positions_for_log = global_positions
        if raw_k.shape[2] == 0:
            out.append({"k": raw_k, "v": raw_v})
            continue
        target_positions = torch.tensor(
            [float(logical_positions_by_global_pos[g]) for g in global_positions]
        )
        rotated_k = apply_rope(raw_k, target_positions, FREQS)
        out.append({"k": rotated_k, "v": raw_v})

    if positions_for_log:
        logger.debug(
            f"rotate_survivors_from_raw: {len(positions_for_log)} survivors, "
            f"global_positions={positions_for_log}, "
            f"target_logical_positions={[logical_positions_by_global_pos[g] for g in positions_for_log]}"
        )
    return out


@torch.no_grad()
def apply_query_side_correction(q, eviction_shift):
    """RFC §3.1's query-side identity -- rotate the QUERY by
    -eviction_shift instead of re-rotating survivor KEYS.

    NOT CALLED in run_strategy_c_rsqr below, and left uncalled
    deliberately. model_step's attention call applies ONE query against
    the FULL concatenated key cache (sink+fact+pinned+survivors+window)
    in a single matmul, so a query rotated specifically to compensate
    for the survivor region's eviction shift would also incorrectly get
    applied against sink/fact/window keys in that same call. True
    per-region query-side correction would require splitting attention
    into multiple matmuls (one per differently-shifted key region) and
    combining softmax-normalized partial outputs -- a real architecture
    change, not implemented in this harness.

    rotate_survivors_from_raw (single-hop key rotation from raw to
    current logical position) is used instead, and is mathematically
    equivalent for this experiment's purposes: RoPE attention scores
    depend only on the RELATIVE position between q and k, so rotating
    a key by -shift produces the same q.k dot product as rotating the
    query by -shift would, for that q-k pair. This function is kept for
    reference / future use if the harness is refactored to per-region
    attention, but is currently dead code -- do not assume it runs.
    """
    shift_t = torch.full((q.shape[2],), float(-eviction_shift))
    corrected = apply_rope(q, shift_t, FREQS)
    logger.debug(f"apply_query_side_correction: shift={eviction_shift}, q_shape={tuple(q.shape)}")
    return corrected


In [ ]:
@torch.no_grad()
def run_strategy_c_rsqr(trial_stream, fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24,
                         block_size=8, survivor_delta=8, trial_tag=""):
    log_prefix = f"[{trial_tag}] " if trial_tag else ""
    sink_caches = empty_caches(N_LAYERS)
    fact_caches = empty_caches(N_LAYERS)
    window_caches = empty_caches(N_LAYERS)
    pinned_caches = empty_caches(N_LAYERS)
    sink_tokens, fact_tokens, pinned_tokens = [], [], []
    window_tokens = []          # list of (tid, global_pos) tuples
    raw_store = RawShadowStore(N_LAYERS)
    survivor_logical_pos = {}   # global_pos -> current logical_pos (compacted, post-eviction)
    cumulative_shift = 0

    # BUG FIX: true_pos is the model's REAL, monotonically-increasing
    # position counter -- every token the model has ever seen gets the
    # next integer, no matter how many evictions have happened. This is
    # what every model_step call for a NEW token must use.
    #
    # logical_pos previously did double duty: it was incremented per
    # token AND decremented by evict_n on every eviction, so it stayed
    # flat across an eviction-heavy run (e.g. stuck at 45 for steps
    # 52/60/68/76 in early debugging) -- meaning every new token after
    # the first eviction was silently assigned the SAME RoPE position as
    # an earlier token. That's a position collision, not a rotation
    # error, and it explains why accuracy specifically collapsed as
    # n_cycles (and eviction count) grew: more evictions meant more
    # tokens compressed onto colliding positions.
    #
    # Fix: true_pos (monotonic, used for model_step) and logical_pos
    # (compacted, used ONLY to compute survivors' target rotation
    # position — logical_pos = true_pos_at_capture - cumulative_shift)
    # are now tracked separately. logical_pos itself is derived, not an
    # independent counter, so it can never drift out of sync with
    # cumulative_shift.
    true_pos = 0
    global_pos = 0
    n_fact_tokens_seen = 0
    n_window_tokens_total = 0
    result = {"correct": None, "answer_logprob": None, "n_evictions": 0,
              "n_survivors_retained_at_query": 0, "n_survivors_flagged_total": 0,
              "n_survivors_evicted_before_query": 0}

    logger.info(f"{log_prefix}run_strategy_c_rsqr START "
                f"sink_size={sink_size} window_size={window_size} "
                f"block_size={block_size} survivor_delta={survivor_delta} "
                f"stream_len={len(trial_stream)} fact_len={fact_len}")

    def build_base(survivor_cache):
        # Concat order: sink -> fact -> pinned -> survivors -> window.
        # Pinned sits right after fact (fixed position, never shifts as
        # window grows/evicts or as survivors are added/rotated) so
        # window's own offset math never has to account for pinned
        # content moving underneath it.
        return [concat_cache(s, concat_cache(f, concat_cache(p, concat_cache(sv, w))))
                for s, f, p, sv, w in zip(sink_caches, fact_caches, pinned_caches,
                                           survivor_cache or empty_caches(N_LAYERS),
                                           window_caches)]

    for step_idx, (tid, is_query, answer_id, is_pinned) in enumerate(trial_stream):
        if is_query:
            survivor_cache = None
            if len(raw_store) > 0:
                survivor_cache = rotate_survivors_from_raw(raw_store, survivor_logical_pos)
            base = build_base(survivor_cache)
            true_pos += 1
            logits, _ = model_step(tid, true_pos, base)
            true_pos -= 1
            result["correct"] = bool(logits.argmax(-1).item() == answer_id)
            result["answer_logprob"] = F.log_softmax(logits, dim=-1)[answer_id].item()
            result["n_survivors_retained_at_query"] = len(raw_store)
            top5 = torch.topk(logits, 5)
            logger.info(
                f"{log_prefix}QUERY step={step_idx} answer_id={answer_id} "
                f"predicted={logits.argmax(-1).item()} correct={result['correct']} "
                f"logprob={result['answer_logprob']:.4f} "
                f"survivors_alive={len(raw_store)} n_evictions_so_far={result['n_evictions']} "
                f"top5_ids={top5.indices.tolist()} top5_logits={[round(v,2) for v in top5.values.tolist()]}"
            )
            break

        global_pos += 1
        true_pos += 1
        survivor_cache = None
        if len(raw_store) > 0:
            survivor_cache = rotate_survivors_from_raw(raw_store, survivor_logical_pos)
        base = build_base(survivor_cache)

        if len(sink_tokens) < sink_size:
            sink_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            sink_caches = [slice_cache(c, start=0, end=len(sink_tokens)) for c in new_full]
            logger.debug(f"{log_prefix}SINK step={step_idx} global_pos={global_pos} "
                         f"true_pos={true_pos} sink_count={len(sink_tokens)}")

        elif n_fact_tokens_seen < fact_len:
            fact_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact_prior = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            fact_caches = [slice_cache(c, start=n_sink, end=n_sink + n_fact_prior + 1) for c in new_full]
            n_fact_tokens_seen += 1
            logger.debug(f"{log_prefix}FACT step={step_idx} global_pos={global_pos} "
                         f"true_pos={true_pos} fact_tokens_seen={n_fact_tokens_seen}/{fact_len}")

        elif is_pinned:
            # Query-prefix token: pinned region only (between fact and
            # survivors/window in concat order). Never counted toward
            # n_window_tokens_total, so it can NEVER be flagged into
            # raw_store, and never enters window_tokens, so it can NEVER
            # be evicted. Plain model_step suffices -- no raw-key capture
            # needed since this token is never a flagging candidate.
            pinned_tokens.append(tid)
            _, new_full = model_step(tid, true_pos, base)
            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            pinned_caches = [slice_cache(c, start=n_sink + n_fact, end=n_sink + n_fact + len(pinned_tokens)) for c in new_full]
            logger.debug(f"{log_prefix}PINNED step={step_idx} global_pos={global_pos} "
                         f"true_pos={true_pos} pinned_count={len(pinned_tokens)}")

        else:
            n_window_tokens_total += 1
            is_flagged = (n_window_tokens_total % survivor_delta == 0)
            logits_ignore, new_full, raw_ks, raw_vs = model_step_with_raw_capture(tid, true_pos, base)

            if is_flagged:
                raw_store.add(global_pos, raw_ks, raw_vs)
                # BUG FIX (rank-based compaction, RFC \u00a73.2 step 4):
                # the previous scheme (survivor_logical_pos[g] = true_pos
                # - cumulative_shift) UNIFORMLY TRANSLATES all survivors
                # by the total evicted count -- it never closes the gaps
                # between them. Two survivors originally 500 tokens apart
                # in the raw stream stay 500 apart forever under that
                # scheme. The RFC is explicit that compaction should
                # produce a CONSECUTIVE logical timeline (global position
                # 505 -> logical position 5) -- i.e. rank among
                # currently-alive survivors, not a shift-preserving
                # subtraction. Fixed here: every alive survivor's logical
                # position is recomputed as its rank (0-indexed, sorted
                # by global_pos ascending) among ALL currently-alive
                # survivors, every time a new one is flagged -- because
                # inserting a new survivor can change the rank of
                # existing ones. Cheap (O(n_survivors), not
                # O(sequence length)) and correct regardless of flagging
                # order or eviction timing.
                _sorted_survivors = sorted(list(survivor_logical_pos.keys()) + [global_pos])
                for _rank, _g in enumerate(_sorted_survivors):
                    survivor_logical_pos[_g] = _rank
                result["n_survivors_flagged_total"] += 1
                logger.info(f"{log_prefix}FLAG step={step_idx} global_pos={global_pos} "
                           f"true_pos={true_pos} logical_pos={survivor_logical_pos[global_pos]} "
                           f"(survivor #{result['n_survivors_flagged_total']}, rank-compacted, "
                           f"n_alive_survivors={len(survivor_logical_pos)})")

                # SEAM ASSERTION: no two alive survivors may share a
                # logical position, and ranks must be exactly
                # 0..n-1 with no gaps or repeats. This is the check that
                # would have caught the shift-based bug immediately
                # instead of requiring a manual multi-round log read --
                # keep this on every run, cost is negligible next to a
                # model forward pass.
                _lp_values = sorted(survivor_logical_pos.values())
                _expected = list(range(len(survivor_logical_pos)))
                assert _lp_values == _expected, (
                    f"{log_prefix}SURVIVOR LOGICAL-POSITION COLLISION/GAP: "
                    f"expected exactly {_expected}, got {_lp_values}. "
                    f"Full map: {survivor_logical_pos}"
                )

            n_sink = sink_caches[0]["k"].shape[2] if sink_caches[0] is not None else 0
            n_fact = fact_caches[0]["k"].shape[2] if fact_caches[0] is not None else 0
            n_pinned = pinned_caches[0]["k"].shape[2] if pinned_caches[0] is not None else 0
            n_surv = len(raw_store)
            window_tokens.append((tid, global_pos))
            window_start = n_sink + n_fact + n_pinned + n_surv
            window_caches = [slice_cache(c, start=window_start, end=window_start + len(window_tokens))
                              for c in new_full]

            logger.debug(f"{log_prefix}WINDOW step={step_idx} global_pos={global_pos} "
                        f"true_pos={true_pos} window_len={len(window_tokens)} "
                        f"flagged={is_flagged} survivors_alive={len(raw_store)}")

            if len(window_tokens) > 0 and len(window_tokens) % block_size == 0 and len(window_tokens) > window_size:
                evict_n = block_size
                evicted = window_tokens[:evict_n]
                evicted_global_positions = [g for (_, g) in evicted]

                window_tokens = window_tokens[evict_n:]
                window_caches = [slice_cache(c, start=evict_n) for c in window_caches]

                n_survivors_in_evicted_batch = sum(
                    1 for g in evicted_global_positions if g in survivor_logical_pos
                )
                result["n_survivors_evicted_before_query"] += (
                    len(evicted_global_positions) - n_survivors_in_evicted_batch
                )

                # BUG FIX (rank-based compaction, matches the FLAG-time
                # fix above): survivor_logical_pos is now RANK among
                # currently-alive survivors, not a shift-preserving
                # subtraction. Eviction removes non-survivor window
                # tokens only (flagged survivors already living in
                # raw_store are untouched by this eviction -- see
                # n_survivors_in_evicted_batch check above, which
                # correctly finds them still present in
                # survivor_logical_pos). Since NO survivor is removed
                # from raw_store by a plain window eviction, the alive
                # survivor SET doesn't change here, and therefore neither
                # does anyone's rank -- ranks only need recomputing when
                # the alive set changes (i.e. at FLAG time, already
                # handled above). No action needed on eviction itself.
                # cumulative_shift is kept only for logging/diagnostics
                # below, not as an input to any rotation target anymore.
                cumulative_shift += evict_n
                # true_pos is NEVER decremented -- it is the model's real,
                # ever-increasing position counter. Only logical_pos
                # (via cumulative_shift) tracks the eviction-induced
                # compaction, and only for survivor rotation targets.
                result["n_evictions"] += 1

                logger.info(
                    f"{log_prefix}EVICT step={step_idx} evict_n={evict_n} "
                    f"evicted_global_positions={evicted_global_positions} "
                    f"survivors_among_evicted={n_survivors_in_evicted_batch} "
                    f"cumulative_shift={cumulative_shift} true_pos={true_pos} "
                    f"survivors_alive={len(raw_store)} (eviction #{result['n_evictions']})"
                )

    logger.info(f"{log_prefix}run_strategy_c_rsqr END result={result} "
                f"shadow_store_summary={raw_store.summary()}")
    return result


## Sanity check before the full sweep

Confirms the stream is 4-tuple with `is_pinned` set correctly, and that
both B and RSQR run cleanly on a small case, before committing to the
full multi-hour sweep below. **If this cell errors or the assertions
fail, stop and don't run the sweep.**


In [ ]:
set_rsqr_log_level(logging.DEBUG)
rng = np.random.default_rng(0)
_stream, _fact_len, _magic = make_multi_cycle_stream_fast(rng, n_cycles=4, cycle_len=16, sink_size=len(SYSTEM_PREFIX_IDS))

assert len(_stream[0]) == 4, f"expected 4-tuples, got {len(_stream[0])}-tuples -- stream generator not updated"
assert any(t[3] for t in _stream), "no token was tagged is_pinned=True -- tag_query_prefix not wired in"
assert _stream[-1][1] is True, "last token should be the is_query token"
assert _stream[-1][3] is True, "the is_query token itself must also be pinned"
print("stream format OK:", len(_stream), "tokens, tail:", _stream[-6:])

r_c, r_u = run_strategy_b_pair_fast(_stream, fact_len=_fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, block_size=8)
r_rsqr = run_strategy_c_rsqr(_stream, _fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, block_size=8,
                              survivor_delta=8, trial_tag="smoketest")
print("B corrected:", r_c)
print("B uncorrected:", r_u)
print("C RSQR:", r_rsqr)
print("\nSmoke test passed -- safe to run the full sweep below.")


rsqr logger level set to DEBUG
stream format OK: 101 tokens, tail: [(13, False, None, True), (576, False, None, True), (6234, False, None, True), (1372, False, None, True), (374, False, None, True), (220, True, 22, True)]
[INFO] [smoketest] run_strategy_c_rsqr START sink_size=24 window_size=24 block_size=8 survivor_delta=8 stream_len=101 fact_len=7
[DEBUG] [smoketest] SINK step=0 global_pos=1 true_pos=1 sink_count=1
[DEBUG] [smoketest] SINK step=1 global_pos=2 true_pos=2 sink_count=2
[DEBUG] [smoketest] SINK step=2 global_pos=3 true_pos=3 sink_count=3
[DEBUG] [smoketest] SINK step=3 global_pos=4 true_pos=4 sink_count=4
[DEBUG] [smoketest] SINK step=4 global_pos=5 true_pos=5 sink_count=5
[DEBUG] [smoketest] SINK step=5 global_pos=6 true_pos=6 sink_count=6
[DEBUG] [smoketest] SINK step=6 global_pos=7 true_pos=7 sink_count=7
[DEBUG] [smoketest] SINK step=7 global_pos=8 true_pos=8 sink_count=8
[DEBUG] [smoketest] SINK step=8 global_pos=9 true_pos=9 sink_count=9
[DEBUG] [smoketest] SINK ste

In [ ]:
print(repr(tokenizer.decode([7578])))

# Direct HF check with the EXACT SAME full token sequence the smoke test just used
# (rebuild via the same generator + seed as the smoke test cell above)
rng_direct = np.random.default_rng(0)
_stream_direct, _fact_len_direct, _magic_direct = make_multi_cycle_stream_fast(
    rng_direct, n_cycles=4, cycle_len=16, sink_size=len(SYSTEM_PREFIX_IDS)
)
full_ids_direct = [tid for tid, _, _, _ in _stream_direct]
answer_id_direct = _stream_direct[-1][2]

inputs_direct = torch.tensor([full_ids_direct], device=device)
with torch.no_grad():
    out_direct = hf_model(input_ids=inputs_direct)
logits_direct = out_direct.logits[0, -1]
top5_direct = torch.topk(logits_direct, 5)
print("magic number:", _magic_direct, "answer_id:", answer_id_direct)
print("top5 ids:", top5_direct.indices.tolist())
print("top5 tokens:", [tokenizer.decode([i]) for i in top5_direct.indices.tolist()])
print("top5 logits:", top5_direct.values.tolist())

' sat'
magic number: 7 answer_id: 22
top5 ids: [22, 16, 23, 24, 20]
top5 tokens: ['7', '1', '8', '9', '5']
top5 logits: [21.095914840698242, 18.18907928466797, 17.560503005981445, 17.377338409423828, 17.313812255859375]


**Historical note:** this cell used to hold the standalone probe that isolated the suffix-format bug (chat-wrapped prefix + plain-completion suffix, confirmed correct@rank1 in 3/3 trials). That fix is now applied directly inside `make_multi_cycle_stream_fast` above, so this cell is no longer needed as separate runnable code.

In [ ]:
def sanitize_row(row):
    sanitized = {}
    for k, v in row.items():
        if isinstance(v, bool):
            sanitized[k] = v
        elif isinstance(v, float):
            sanitized[k] = None if math.isnan(v) else v
        elif hasattr(v, "item"):
            sanitized[k] = v.item()
        elif isinstance(v, (np.integer, np.int64)):
            sanitized[k] = int(v)
        else:
            sanitized[k] = v
    return sanitized


In [ ]:
set_rsqr_log_level(logging.DEBUG)

rsqr logger level set to DEBUG


## Sweep: B (corrected/uncorrected) + RSQR only

Strategy A dropped from this run since it's not part of the
leave-gap-vs-RSQR crossover question and its replay/consolidation path
is the slowest part of the sweep -- removing it should meaningfully
shorten the run relative to the previous 4-way version.


In [ ]:
from tqdm import tqdm

set_rsqr_log_level(logging.INFO)   # bump to logging.DEBUG for a full trace on a small run first

#LOG_PATH_BC = "per_trial_log_b_rsqr.jsonl"
#open(LOG_PATH_BC, "w").close()

CYCLE_LEN = 16
N_TRIALS_PER_K = 20
SEEDS = [0, 1, 2]
cycle_values = [2, 4, 6, 12, 16, 32, 64]
SURVIVOR_DELTA = 8   # RFC §3.2 step 2 -- every Delta-th window token flagged

summary_bc = []
per_trial_log_bc = []

for n_cycles in tqdm(cycle_values, desc="Sweeping Cycle Values (B + RSQR)"):
    trial_results = []
    for seed in SEEDS:
        rng = np.random.default_rng(seed * 1000 + n_cycles)
        for trial_i in range(N_TRIALS_PER_K):
            stream, fact_len, magic_number = make_multi_cycle_stream_fast(rng, n_cycles, CYCLE_LEN, sink_size=len(SYSTEM_PREFIX_IDS))
            tag = f"seed{seed}_cycles{n_cycles}_trial{trial_i}"

            r_c, r_u = run_strategy_b_pair_fast(
                stream, fact_len=fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, block_size=8
            )
            r_rsqr = run_strategy_c_rsqr(
                stream, fact_len, sink_size=len(SYSTEM_PREFIX_IDS), window_size=24, block_size=8,
                survivor_delta=SURVIVOR_DELTA, trial_tag=tag,
            )

            rsqr_exercised = r_rsqr["n_survivors_flagged_total"] > 0 and r_rsqr["n_evictions"] > 0
            if not rsqr_exercised:
                logger.warning(
                    f"[{tag}] RSQR path NOT exercised this trial "
                    f"(flagged={r_rsqr['n_survivors_flagged_total']}, "
                    f"evictions={r_rsqr['n_evictions']}) -- accuracy for this "
                    f"row does not test the mechanism, mark separately downstream"
                )

            row = {
                "n_cycles": n_cycles, "seed": seed, "magic_number": magic_number,
                "correct_B_corrected": r_c["correct"], "logprob_B_corrected": r_c["answer_logprob"],
                "correct_B_uncorrected": r_u["correct"], "logprob_B_uncorrected": r_u["answer_logprob"],
                "correct_C_rsqr": r_rsqr["correct"], "logprob_C_rsqr": r_rsqr["answer_logprob"],
                "n_evictions_C_rsqr": r_rsqr["n_evictions"],
                "n_survivors_flagged_C_rsqr": r_rsqr["n_survivors_flagged_total"],
                "n_survivors_retained_at_query_C_rsqr": r_rsqr["n_survivors_retained_at_query"],
                "rsqr_path_exercised": rsqr_exercised,
            }
            trial_results.append(row)
            sanitized = sanitize_row(row)
            per_trial_log_bc.append(sanitized)
            with open(LOG_PATH_BC, "a") as f:
                f.write(json.dumps(sanitized) + "\n")

    exercised_rows = [r for r in trial_results if r["rsqr_path_exercised"]]
    acc_c = np.mean([r["correct_B_corrected"] for r in trial_results])
    acc_u = np.mean([r["correct_B_uncorrected"] for r in trial_results])
    acc_rsqr = np.mean([r["correct_C_rsqr"] for r in exercised_rows]) if exercised_rows else float("nan")
    n_exercised = len(exercised_rows)

    summary_bc.append((n_cycles, acc_c, acc_u, acc_rsqr, n_exercised))
    print(f"n_cycles={n_cycles:3d} | B corrected: {acc_c*100:5.1f}% | B uncorrected: {acc_u*100:5.1f}% | "
          f"C RSQR: {acc_rsqr*100:5.1f}% (n_exercised={n_exercised}/{len(trial_results)})")

print(f"\nTotal trials logged: {len(per_trial_log_bc)}")


rsqr logger level set to INFO


Sweeping Cycle Values (B + RSQR):   0%|          | 0/7 [00:00<?, ?it/s]

[INFO] [seed0_cycles2_trial0] run_strategy_c_rsqr START sink_size=24 window_size=24 block_size=8 survivor_delta=8 stream_len=69 fact_len=7
[INFO] [seed0_cycles2_trial0] FLAG step=38 global_pos=39 true_pos=39 logical_pos=0 (survivor #1, rank-compacted, n_alive_survivors=1)
[INFO] [seed0_cycles2_trial0] FLAG step=46 global_pos=47 true_pos=47 logical_pos=1 (survivor #2, rank-compacted, n_alive_survivors=2)
[INFO] [seed0_cycles2_trial0] FLAG step=54 global_pos=55 true_pos=55 logical_pos=2 (survivor #3, rank-compacted, n_alive_survivors=3)
[INFO] [seed0_cycles2_trial0] FLAG step=62 global_pos=63 true_pos=63 logical_pos=3 (survivor #4, rank-compacted, n_alive_survivors=4)
[INFO] [seed0_cycles2_trial0] EVICT step=62 evict_n=8 evicted_global_positions=[32, 33, 34, 35, 36, 37, 38, 39] survivors_among_evicted=1 cumulative_shift=8 true_pos=63 survivors_alive=4 (eviction #1)
[INFO] [seed0_cycles2_trial0] QUERY step=68 answer_id=22 predicted=22 correct=True logprob=-1.2277 survivors_alive=4 n_evict

Sweeping Cycle Values (B + RSQR):  14%|█▍        | 1/7 [05:37<33:42, 337.06s/it]

n_cycles=  2 | B corrected:  93.3% | B uncorrected:  93.3% | C RSQR:  80.0% (n_exercised=60/60)
[INFO] [seed0_cycles4_trial0] run_strategy_c_rsqr START sink_size=24 window_size=24 block_size=8 survivor_delta=8 stream_len=101 fact_len=7
[INFO] [seed0_cycles4_trial0] FLAG step=38 global_pos=39 true_pos=39 logical_pos=0 (survivor #1, rank-compacted, n_alive_survivors=1)
[INFO] [seed0_cycles4_trial0] FLAG step=46 global_pos=47 true_pos=47 logical_pos=1 (survivor #2, rank-compacted, n_alive_survivors=2)
[INFO] [seed0_cycles4_trial0] FLAG step=54 global_pos=55 true_pos=55 logical_pos=2 (survivor #3, rank-compacted, n_alive_survivors=3)
[INFO] [seed0_cycles4_trial0] FLAG step=62 global_pos=63 true_pos=63 logical_pos=3 (survivor #4, rank-compacted, n_alive_survivors=4)
[INFO] [seed0_cycles4_trial0] EVICT step=62 evict_n=8 evicted_global_positions=[32, 33, 34, 35, 36, 37, 38, 39] survivors_among_evicted=1 cumulative_shift=8 true_pos=63 survivors_alive=4 (eviction #1)
[INFO] [seed0_cycles4_trial

...

n_cycles= 12 | B corrected: 100.0% | B uncorrected:  28.3% | C RSQR:  78.3% (n_exercised=60/60)
[INFO] [seed0_cycles16_trial0] run_strategy_c_rsqr START sink_size=24 window_size=24 block_size=8 survivor_delta=8 stream_len=293 fact_len=7
[INFO] [seed0_cycles16_trial0] FLAG step=38 global_pos=39 true_pos=39 logical_pos=0 (survivor #1, rank-compacted, n_alive_survivors=1)
[INFO] [seed0_cycles16_trial0] FLAG step=46 global_pos=47 true_pos=47 logical_pos=1 (survivor #2, rank-compacted, n_alive_survivors=2)
[INFO] [seed0_cycles16_trial0] FLAG step=54 global_pos=55 true_pos=55 logical_pos=2 (survivor #3, rank-compacted, n_alive_survivors=3)
[INFO] [seed0_cycles16_trial0] FLAG step=62 global_pos=63 true_pos=63 logical_pos=3 (survivor #4, rank-compacted, n_alive_survivors=4)
[INFO] [seed0_cycles16_trial0] EVICT step=62 evict_n=8 evicted_global_positions=[32, 33, 34, 35, 36, 37, 38, 39] survivors_among_evicted=1 cumulative_shift=8 true_pos=63 survivors_alive=4 (eviction #1)
[INFO] [seed0_cycles1

Sweeping Cycle Values (B + RSQR):  71%|███████▏  | 5/7 [1:20:39<41:16, 1238.44s/it]

Streaming output truncated to the last 5000 lines.
[INFO] [seed1_cycles32_trial0] FLAG step=526 global_pos=527 true_pos=527 logical_pos=61 (survivor #62, rank-compacted, n_alive_survivors=62)
[INFO] [seed1_cycles32_trial0] EVICT step=526 evict_n=8 evicted_global_positions=[496, 497, 498, 499, 500, 501, 502, 503] survivors_among_evicted=1 cumulative_shift=472 true_pos=527 survivors_alive=62 (eviction #59)
[INFO] [seed1_cycles32_trial0] FLAG step=534 global_pos=535 true_pos=535 logical_pos=62 (survivor #63, rank-compacted, n_alive_survivors=63)
[INFO] [seed1_cycles32_trial0] EVICT step=534 evict_n=8 evicted_global_positions=[504, 505, 506, 507, 508, 509, 510, 511] survivors_among_evicted=1 cumulative_shift=480 true_pos=535 survivors_alive=63 (eviction #60)
[INFO] [seed1_cycles32_trial0] FLAG step=542 global_pos=543 true_pos=543 logical_pos=63 (survivor #64, rank-compacted, n_alive_survivors=64)
[INFO] [seed1_cycles32_trial0] EVICT step=542 evict_n=8 evicted_global_positions=[512, 513, 51

Sweeping Cycle Values (B + RSQR):  86%|████████▌ | 6/7 [2:22:06<34:30, 2070.87s/it]

Streaming output truncated to the last 5000 lines.
[INFO] [seed1_cycles64_trial2] FLAG step=526 global_pos=527 true_pos=527 logical_pos=61 (survivor #62, rank-compacted, n_alive_survivors=62)
[INFO] [seed1_cycles64_trial2] EVICT step=526 evict_n=8 evicted_global_positions=[496, 497, 498, 499, 500, 501, 502, 503] survivors_among_evicted=1 cumulative_shift=472 true_pos=527 survivors_alive=62 (eviction #59)
[INFO] [seed1_cycles64_trial2] FLAG step=534 global_pos=535 true_pos=535 logical_pos=62 (survivor #63, rank-compacted, n_alive_survivors=63)
[INFO] [seed1_cycles64_trial2] EVICT step=534 evict_n=8 evicted_global_positions=[504, 505, 506, 507, 508, 509, 510, 511] survivors_among_evicted=1 cumulative_shift=480 true_pos=535 survivors_alive=63 (eviction #60)
[INFO] [seed1_cycles64_trial2] FLAG step=542 global_pos=543 true_pos=543 logical_pos=63 (survivor #64, rank-compacted, n_alive_survivors=64)
[INFO] [seed1_cycles64_trial2] EVICT step=542 evict_n=8 evicted_global_positions=[512, 513, 51

## Optional: mount Drive and copy the log there

Run this BEFORE the sweep cell above if you want the log to survive a
disconnect -- swap `LOG_PATH_BC` for the Drive path first.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = "/content/drive/MyDrive/colab_experiments"
os.makedirs(DRIVE_DIR, exist_ok=True)

LOG_PATH_BC = os.path.join(DRIVE_DIR, "per_trial_log_b_rsqr.jsonl")
print(f"Log file mapped to Google Drive: {LOG_PATH_BC}")


Mounted at /content/drive
Log file mapped to Google Drive: /content/drive/MyDrive/colab_experiments/per_trial_log_b_rsqr.jsonl
